**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# LLMs from the Ground Up

Every piece of a modern language model, built small enough to train on a laptop during the session: tokenization, embeddings, the next-token objective, and the inference tricks (temperature, top-k, KV caching) that turn a trained network into a chatbot's engine. The [transformer architecture itself](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb) is a prerequisite — here we focus on the *language modeling* around it.

## 1. Pre-requisites

- [Intro to Transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb) — attention.
- [Training Dynamics](./Training_Dynamics.ipynb) — Adam, schedules.
- [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) — cross-entropy: an LLM is literally trained to *compress text*.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fn
torch.manual_seed(0)

# our corpus: a tiny fable, repeated with variations — small enough to learn in minutes
base = (
"the fox watched the river. the river carried leaves and light. "
"a heron stood in the shallows and waited for fish. "
"the fox wanted fish too, but the fox could not wade. "
"so the fox watched the heron, and the heron watched the water. "
"when the fish rose, the heron struck. the fox learned patience from the heron. "
"in the morning the river was silver. in the evening the river was gold. "
"the leaves drifted, the light faded, and the fox went home with an idea. "
)
text = base * 40                      # ~11k characters
print(f"corpus: {len(text):,} characters")

---
### 🕐 Session 1 of 4 — *Tokens & Embeddings* (~35 min)
**Goal:** turn text into integers, integers into vectors; understand what BPE buys real models.
**Builds on:** [Transformers workshop](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb). &nbsp; **Feeds into:** Session 2 (the pretraining objective).

---

## 2. Text → Numbers

💡 **Intuition.** A model eats vectors, not letters. Step 1 — **tokenize**: chop text into pieces from a fixed vocabulary and number them. We use characters (simple, small vocab); real LLMs use **BPE** — start from characters, repeatedly merge the most frequent adjacent pair ('t'+'h'→'th', 'th'+'e'→'the'), until common words are single tokens and rare words split into parts. It's a *compression* scheme ([Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb)!): frequent strings get short codes. Step 2 — **embed**: a learned lookup table maps each token id to a vector; during training, tokens used similarly drift together.

In [ ]:

# YOUR CODE HERE


In [ ]:
# mini-BPE, 12 merges, to see the mechanism real tokenizers scale up

# YOUR CODE HERE


**What just happened.** Twelve merges, chosen by frequency alone, and look at what they built:

`he` → `the` → `the ` → ` the ` → ` w` → ` wa` → `ve` → `fo` → `d ` → `ro` → `fox` → `ri`

**The algorithm was never told about English.** It counted adjacent pairs and merged the most common one, twelve times. Out came the digraph `he`, then the definite article `the`, then the article-with-space ` the `, then the word `fox`. **Frequency alone recovered linguistic units** — and that is the entire idea behind every production tokeniser, scaled from 12 merges to about 50,000.

**Notice that BPE is a compression algorithm and that this is not a metaphor.** Frequent strings receive short codes; rare ones stay decomposed. That is the source-coding principle from [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) applied to text, and it is why the tokenised sequence is shorter than the character sequence. **Tokenisation and compression are the same operation with different downstream uses.**

**The spaces in ` the ` and `d ` are not a bug, and they explain real LLM behaviour.** BPE merges across word boundaries because spaces are frequent characters, so tokens routinely carry leading or trailing whitespace. This is why **token counts do not equal word counts**, why `"the"` and `" the"` are distinct tokens in GPT models, and why prompts sometimes behave differently depending on trailing spaces.

**And it is the root of a well-known class of LLM failures.** After tokenisation the model sees `['the ', 'fox', ' wa', 't', 'c', 'he', 'd']` — **not letters**. Asking a model to count the r's in "strawberry" or reverse a string asks about a representation it does not have; the characters were merged away before the network saw anything. **Arithmetic suffers for the same reason**: whether "1234" is one token or three depends on the merge table, and the model must learn digit structure through an encoding that scrambles it.

**Note the inconsistency the sample tokenisation reveals.** `'the '` appears as one token in one place and ` the ` in another, and `watched` splits as `' wa', 't', 'c', 'he', 'd'` — the same word can tokenise differently depending on what precedes it. **BPE is greedy and left-to-right, not optimal**, and that irregularity is present in every production tokeniser too.

**One honest limitation of this miniature.** Twelve merges on a 454-character passage produces merges specific to *this fable* — `fox` and `ri` are frequent here and nowhere else. Real tokenisers are fitted on hundreds of gigabytes, which is why their vocabularies generalise. **The mechanism is identical; the statistics are what make it useful.**

---
### 🕐 Session 2 of 4 — *Pretraining: the Next-Token Objective* (~40 min)
**Goal:** train a small GPT on next-character prediction; watch loss approach the corpus entropy.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (finetuning at a glance).

---

## 3. One Objective To Rule Them All

💡 **Intuition.** The entire pretraining recipe is: *predict the next token, everywhere, forever*. The loss is cross-entropy — so training literally minimizes the bits needed to encode the corpus ([source coding](../Intro_Math/Information_Theory/Information_Theory.ipynb)): **an LLM is a learned compressor**, and everything it 'knows' exists because knowing it helps compression. Grammar helps predict; facts help predict; style helps predict. Scale the corpus and the model, and the compressor is forced to become a world-modeler.

In [ ]:

# YOUR CODE HERE


In [ ]:
# corpus entropy baselines: what loss SHOULD we expect?

# YOUR CODE HERE


In [ ]:

# YOUR CODE HERE


**What just happened.** Loss fell from **3.601 nats** — essentially the $\ln 25 = 3.219$ of uniform guessing, plus initialisation noise — to **0.067 nats** in 600 steps. Against the unigram entropy of 2.763, that is a 40× reduction.

**Having the two baselines printed beforehand is what makes this readable at all.** Uniform guessing is 3.219; letter frequencies alone get you 2.763; the model reached 0.067. **Without those numbers "loss 0.067" is an uninterpretable decimal**, and computing your own baselines before training is the habit worth taking from this cell.

**But there is a third baseline the notebook does not print, and it changes the verdict.** The corpus is `base * 40` — **the same 454-character passage repeated forty times**. Given a 64-character context window, every character after the first repetition is *perfectly determined* by what precedes it. **The true conditional entropy of this corpus is essentially zero.** So 0.067 nats is not "learned real structure"; it is **memorisation**, and a perfect memoriser would score lower still.

**The printed conclusion therefore overstates the result, and the honest version is more useful.** What the model demonstrably learned is that this text is highly predictable, which it is — by construction. **Beating unigram entropy on a corpus with essentially no entropy is not evidence of language modelling.** It is evidence that the training loop, the causal mask, and the optimiser all work, which is a real and necessary thing to establish.

**The five-minute experiment that settles it: hold out a copy.** Train on `base * 39`, validate on the remaining copy, and report both. Here they would agree, because the validation copy is identical to the training data — which is itself the demonstration. **The real test is a corpus with genuine novelty**, and adding a second, different fable would make the train/validation gap immediately visible.

**What rescues the demo is that the regime, not the mechanism, is wrong.** At scale the corpus is not repeated, memorisation is impossible, and the pressure to compress is exactly what forces general structure to appear. **Grammar helps predict; facts help predict; style helps predict** — that is the honest reason a next-token objective produces something that looks like understanding, and it operates precisely when memorisation is unavailable. [Scale_NN](./Scale_NN/Scale_NN.ipynb) made the same point from the other direction.

**Note the objective's data efficiency while it is on screen, because it is genuinely remarkable.** Each batch is 32 sequences of 64 tokens, and **every position is a supervised example** — 2,048 predictions per step, all from unlabelled text. The causal mask is what makes that honest: position $t$ cannot see beyond $t$, so each of the 64 predictions is a real forecast rather than a lookup. **No human labelled anything**, and that is why pretraining scales to the entire internet.

**Finally, the loss units are worth one sentence.** Cross-entropy in nats *is* the average code length, so 0.067 nats ≈ 0.097 bits per character — the model would compress this text roughly 80× better than 8-bit ASCII. **Training a language model and fitting a compressor are the same optimisation**, which is the framing [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) supplies and which makes the whole objective feel less arbitrary.

In [ ]:

# YOUR CODE HERE


**What just happened.** A 107k-parameter model, trained for under a minute, produced fluent, grammatical, correctly-punctuated English — complete with the fable's characters, its sentence rhythm, and its full stops in the right places. **From nothing but "predict the next character."**

**Read the output carefully, though, because it is closer to recitation than to generation.** Compare it against `base`: the generated text follows the source almost verbatim, in order. That is what a corpus of **one passage repeated forty times** produces — the model learned the sequence, and sampling replays it. **Fluency here is memorisation wearing a convincing costume.**

**There is one visible seam worth pointing at: "the river r carried leaves".** A stray `r` where the source has none. That single character is the model *not* being a lookup table — a small imperfection in the memorisation, surfacing because sampling at temperature 0.8 occasionally departs from the argmax. **Genuine artefacts are more informative than clean output**, and this one shows the mechanism is probabilistic rather than a replay buffer.

**Note what the model was never told and nonetheless reproduces.** Nobody supplied a vocabulary, a grammar, a list of characters, or a rule that sentences end in periods. All of it is a consequence of one objective — **predict the next token** — applied to text where those regularities pay off in compression. That is the honest version of "structure emerges from prediction", and it holds even in this degenerate corpus.

**The generation loop itself is worth reading line by line, because it is the whole of inference.** Take the last `block` tokens; run the model; take the final position's logits; divide by temperature; optionally truncate to top-$k$; sample; append; repeat. **Six lines, and it is exactly what a production chatbot does** — everything else is batching, caching, and safety filtering.

**Point at `idx[:, -model.block:]` as the constraint it is.** The model's context window is **64 characters**. Anything earlier is simply gone — not down-weighted, not summarised, gone. Generate 500 tokens and the model has no memory of how it started. **Context length is a hard wall, not a soft preference**, and extending it is precisely what the quadratic-attention problem in [Modern Architectures](./Modern_Architectures.ipynb) makes expensive.

**Finally, keep the scale in view.** 107k parameters is six orders of magnitude below GPT-3's 175 billion, and the training loop, the objective, and this sampling function are **the same**. What changes with scale is that the corpus stops being memorisable — and at that point the compression pressure has to be paid for with actual structure rather than recall.

---
### 🕐 Session 3 of 4 — *Finetuning & Alignment, at a Glance* (~30 min)
**Goal:** from raw predictor to assistant: SFT, preference learning, and what they change.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (inference).

---

## 4. From Predictor to Assistant

A pretrained model only continues text. Turning it into an assistant is *further training with different data*:

1. **Supervised finetuning (SFT)** — same next-token loss, but on curated (instruction → good response) pairs. The model learns the *format* of being helpful.
2. **Preference tuning (RLHF/DPO)** — humans rank pairs of responses; the model is pushed toward preferred ones. This shapes *judgment*, not knowledge: pretraining knows, alignment chooses.

💡 **Intuition.** Pretraining is the library; finetuning is the librarian's training. Both use gradient descent; only the data — and therefore what's being compressed — changes. We can demo the *mechanism* in miniature: finetune our fable model on a different style and watch the voice change.

In [ ]:
# 'SFT' in miniature: continue training on a new style — terse telegrams

# YOUR CODE HERE


**What just happened.** Two hundred steps on a different corpus, and the model's voice changed completely — from flowing fable prose to clipped telegrams: *"fox waits. heron strikes. fish gone."* **Same weights, same objective, same optimiser. Only the data changed.**

**That is the entire mechanism behind SFT, and it is worth stating in one sentence.** Supervised finetuning uses the **identical** next-token loss; what differs is the data — curated (instruction → good response) pairs instead of raw text. **Pretraining is the library; finetuning is the librarian's training**, and both are gradient descent on cross-entropy. Nothing new is added; a different thing is being compressed.

**But the failure the printout names is the more important result, so give it weight.** The model **forgot** the fable. Prompt it with "the fox " and you get telegrams, not the river and the heron. That is **catastrophic forgetting**: 200 steps of unconstrained gradient descent on new data overwrote the old capability entirely. **Finetuning is not additive** — nothing in the loss preserved what the model already knew.

**Which explains a large slice of practical LLM engineering.** Low learning rates and few epochs, so the weights move less. **LoRA and adapters**, which freeze the base model and train a small low-rank addition. **Data mixing**, replaying pretraining data alongside the new task. And in RLHF, the **KL anchor** — an explicit penalty for drifting from the pretrained model. All four are answers to what this cell just demonstrated in 200 steps.

**Note that the demo is a deliberately extreme case, and say why.** Same learning rate as pretraining ($3\times10^{-3}$), a tiny model with no spare capacity, and a new corpus with **no overlap in style** — every ingredient maximises forgetting. Real finetuning uses $10^{-5}$ to $10^{-6}$ and a fraction of an epoch. **The effect is real and this is its worst case**, which is the right way to show a failure mode but the wrong way to estimate its size.

**Flag one detail about the optimiser that a careful student will catch.** The same `opt` object is reused, so **Adam's momentum and second-moment estimates carry over from pretraining** — the finetune inherits an optimiser state fitted to a different loss surface. In practice that usually means larger-than-intended early steps, which contributes to the forgetting seen here. Reinitialising the optimiser is standard, and its absence is a small realism gap.

**Finally, distinguish the two stages of alignment, since the notebook is careful about it and students conflate them.** SFT teaches the **format** of being helpful — the shape of an answer. Preference tuning (RLHF/DPO) shapes **judgment** — which of two acceptable answers is better. **Pretraining knows; alignment chooses.** Neither adds facts, and the [Reinforcement Learning](./Reinforcement_Learning.ipynb) workshop supplies the machinery for the second.

---
### 🕐 Session 4 of 4 — *Inference: Sampling & the KV Cache* (~35 min)
**Goal:** temperature and top-k as knobs on a distribution; why caching makes generation O(1) per token.
**Builds on:** Sessions 2–3.

---

## 5. Serving the Model

💡 **Intuition.** **Sampling knobs.** The model outputs a *distribution*; how you draw from it sets the personality. Temperature $T$ rescales logits before softmax: $T \to 0$ is argmax (deterministic, repetitive), $T > 1$ flattens (creative, error-prone). Top-k truncates to the $k$ most likely before sampling — a guardrail against the long tail of nonsense.

**The KV cache.** Naive generation re-runs the whole prefix for every new token — $O(n^2)$ pain. But causal attention means old tokens' keys/values *never change*: cache them, and each new token costs one attention row. This single trick is why chatbots stream tokens at constant speed — and why long contexts eat GPU memory (the cache IS the memory hog).

In [ ]:

# YOUR CODE HERE


**What just happened.** Three temperatures — 0.3, 0.8, 1.5 — and **three identical outputs**. Character for character. The knob did nothing.

**That is not a rendering glitch and it should not be waved past.** Temperature divides the logits before the softmax, so it can only *rescale* uncertainty that already exists. After 200 finetuning steps on a 62-character string repeated 60 times, this model's next-token distribution is essentially **one-hot** — logit gaps of tens of nats. Dividing a gap of 40 by 1.5 leaves 27, and softmax of 27 is still indistinguishable from 1.0. **There is no uncertainty for temperature to act on.**

**So the demo demonstrates the opposite of its intent, and the failure is the lesson.** **A model with no entropy has no temperature response.** That is precisely why heavily overfit models produce degenerate, looping text at every setting, and why practitioners reaching for sampling knobs on a memorising model find them inert. The knob is fine; the distribution is degenerate.

**The fix is one line and worth running.** Call this cell on the model **before** the Session 3 finetune, or retrain on a corpus that is not a repeated string. With genuine uncertainty in the distribution, $T = 0.3$ produces rigid near-argmax text, $T = 0.8$ reads naturally, and $T = 1.5$ starts inventing spellings — the three regimes the cell is trying to show.

**What the knobs actually do, since the output cannot show it here.** As $T \to 0$ sampling becomes argmax: deterministic, repetitive, prone to loops. As $T > 1$ the distribution flattens: varied, and increasingly willing to emit tokens the model considers unlikely. **Top-$k$ attacks a different problem** — the long tail. Any single implausible token has tiny probability, but there are thousands of them and their *combined* mass is not negligible, so occasionally one gets drawn. Truncating to the top $k$ removes the tail wholesale rather than down-weighting it.

**Note the framing that survives the failed demo, because it is the important one.** The trained weights define a distribution; **generation is a decoding policy layered on top at serving time**. The same model serves code completion at $T = 0.2$ and brainstorming at $T = 1.0$ without retraining. **Temperature and top-$k$ are not properties of the model** — which is why every inference API exposes them and no training script does.

**One more reason this cell is worth keeping despite not working.** A repetitive, deterministic output at every temperature is a **diagnostic signature**: it says the model has collapsed onto memorised text. If you see it in your own work, the problem is the training data or the amount of finetuning, not the sampler. **Recognising the symptom is more useful than seeing a clean demo.**

In [ ]:
# measure the quadratic blowup the KV cache exists to kill (our model recomputes the prefix)

# YOUR CODE HERE


**What just happened.** Three generation runs, and the per-token cost came out at **0.8 ms/token at every length** — 50, 100, and 200 tokens all identical. The printed conclusion says *"rising, not constant!"*

**It is not rising. The data says constant, and the printed claim contradicts it.** Look at the numbers: 0.04 s / 50, 0.08 s / 100, 0.16 s / 200 — perfectly linear total time, therefore flat per-token cost. **The cell did not measure the quadratic blowup it set out to measure.**

**The reason is one slice in the generation loop: `idx[:, -model.block:]`.** The context is truncated to the last **64** characters, so once generation passes 64 tokens the prefix stops growing and every subsequent step re-encodes exactly 64 positions. **A model with a fixed 64-token window cannot exhibit $O(n^2)$ generation cost**, because $n$ is capped at 64 by construction.

**So the underlying claim is right and the experiment cannot see it.** Naive generation without a cache *is* quadratic: each new token re-runs attention over the whole prefix, so $n$ tokens cost $\sum_{t=1}^{n} O(t) = O(n^2)$. The demo simply removed the growing prefix before timing it. **A measurement that contradicts its own conclusion is worth more than one that confirms it** — provided somebody notices, which is the skill this debrief is for.

**The fix is a two-line exercise and it is the best one in the session.** Raise `block` to 512 (or drop the truncation) and re-time at 50/100/200 tokens. The per-token cost should climb visibly. **Then implement the cache and watch it flatten again** — three measurements, one clean before-and-after, and a claim that is now yours rather than the notebook's.

**The cache argument itself is worth stating precisely, because it is a consequence of causality.** In a causal transformer, token $t$ attends only to positions $\le t$, so the keys and values computed for earlier tokens **can never change** — nothing later influences them. Cache them, and each new token requires one new key/value pair plus one attention row against the cache: $O(n)$ total instead of $O(n^2)$, constant work per token.

**And note the bill, because it is the dominant memory cost in production serving.** The cache holds keys and values for **every layer, every head, every past token**. For a 70B model at 100k context that is tens of gigabytes — often more than the weights. **The KV cache is why long context is expensive at inference time**, and it is what grouped-query and multi-query attention in [Modern Architectures](./Modern_Architectures.ipynb) exist to shrink. Those methods change no mathematics; they share key/value heads so the cache is smaller.

**One last practical note on the measurement itself.** These are single unwarmed runs on CPU at 0.8 ms/token, so the absolute values carry perhaps ±50% and the timing includes Python overhead that would dominate a small model anyway. **For a real comparison, warm up first and average over several runs** — the same discipline the [Scale_NN](./Scale_NN/Scale_NN.ipynb) throughput sweep applies and this cell skips.

## 6. Conclusion

Tokenize (compressively), embed, predict-the-next-token until the loss approaches the corpus's entropy, finetune to choose a voice, then sample with temperature/top-k behind a KV cache. Everything else about LLMs is *scale* — which you studied in [Scaling Neural Networks](./Scale_NN/Scale_NN.ipynb).

---
## Where next

- [Intro to Transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb) — the architecture inside `self.blocks`.
- [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) — the compression view, formalized.
- [Model Compression](./Model_Compression.ipynb) — fitting these onto real hardware.